# CHUYÊN ĐỀ: XỬ LÝ SỰ KIỆN XÓA VẬT LÝ (HARD DELETE) TỪ NGUỒN OLTP TRONG LAKEHOUSE

Trong thực tế, khi bảng nguồn Oracle chạy lệnh:
```sql
DELETE FROM customers WHERE customer_id = 'CUST_101';
```
Hệ thống CDC (Debezium/Oracle GoldenGate) sẽ không gửi payload chứa dữ liệu mới, mà bắn vào Kafka một sự kiện có mã thao tác **`op = "d"`** kèm dữ liệu trước khi xóa (`before`).

Notebook này hướng dẫn **2 Chiến Lược Xử Lý Chuẩn Enterprise**:
1. **Chiến lược 1: Soft Delete (Xóa mềm - Khuyên dùng 90%):** Giữ lại bản ghi nhưng đánh dấu `is_deleted = TRUE` để không làm đứt gãy khóa ngoại của các bảng Fact (đơn hàng cũ).
2. **Chiến lược 2: Hard Delete (Xóa vật lý bằng Delta Merge):** Dùng `.whenMatchedDelete()` để xóa hẳn dòng ra khỏi Delta Lake (tuân thủ luật bảo vệ dữ liệu GDPR/Right to be Forgotten).
3. **Tương tác với tầng Gold (`wh_gold`):** Cách tầng Gold lọc dữ liệu sạch.

## BƯỚC 1: KHỞI TẠO BẢNG SILVER BAN ĐẦU CÓ CỘT `is_deleted` TRÊN `lh_silver.dbo`

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable
from datetime import datetime

LH_BRONZE = "lh_bronze.dbo"
LH_SILVER = "lh_silver.dbo"

# 1. Khởi tạo bảng Silver Customer Value ban đầu (có 2 khách hàng: CUST_101 và CUST_102)
silver_init_data = [
    ("CUST_101", "Nguyen Van An", "Hanoi", 15000000.0, False, None, datetime(2026, 1, 1, 8, 0, 0)),
    ("CUST_102", "Tran Thi Binh", "Danang", 20000000.0, False, None, datetime(2026, 1, 1, 8, 0, 0))
]
silver_schema = StructType([
    StructField("customer_id", StringType(), False),
    StructField("customer_name", StringType(), True),
    StructField("address", StringType(), True),
    StructField("credit_limit", DoubleType(), True),
    StructField("is_deleted", BooleanType(), False),      # Đánh dấu trạng thái xóa
    StructField("deleted_at", TimestampType(), True),     # Thời điểm bị xóa ở nguồn
    StructField("_ingest_at", TimestampType(), False)     # Thời điểm nạp vào Silver
])

spark.createDataFrame(silver_init_data, silver_schema) \
    .write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{LH_SILVER}.customers_delete_demo")

print(f"=== BẢNG {LH_SILVER}.customers_delete_demo BAN ĐẦU ===")
spark.table(f"{LH_SILVER}.customers_delete_demo").show(truncate=False)


## BƯỚC 2: GIẢ LẬP SỰ KIỆN XÓA VẬT LÝ TỪ ORACLE CDC ĐỔ VÀO `lh_bronze.dbo`

Khi `DELETE FROM customers WHERE customer_id = 'CUST_101';` xảy ra ở Oracle:
- Debezium bắn JSON vào Kafka: `{"op": "d", "before": {"customer_id": "CUST_101"}, "ts_ms": 1767258000000}`
- Tầng Bronze tiếp nhận nguyên si chuỗi JSON này.

In [ ]:
# Giả lập tin nhắn CDC từ Kafka đổ vào Bronze Lakehouse
cdc_raw_events = [
    # Sự kiện 1: Sửa thông tin khách CUST_102 (op = u)
    ("evt_01", '{"op": "u", "customer_id": "CUST_102", "customer_name": "Tran Thi Binh", "address": "Hue", "credit_limit": 25000000.0, "cdc_time": "2026-01-01 10:00:00"}'),
    # Sự kiện 2: XÓA VẬT LÝ KHÁCH HÀNG CUST_101 TRÊN ORACLE (op = d)
    ("evt_02", '{"op": "d", "customer_id": "CUST_101", "customer_name": null, "address": null, "credit_limit": null, "cdc_time": "2026-01-01 10:30:00"}')
]

df_cdc_bronze = spark.createDataFrame(cdc_raw_events, ["event_id", "raw_payload"])
df_cdc_bronze.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{LH_BRONZE}.cdc_delete_events")

print(f"=== BẢNG {LH_BRONZE}.cdc_delete_events TIẾP NHẬN SỰ KIỆN CDC (CÓ OP = 'd') ===")
spark.table(f"{LH_BRONZE}.cdc_delete_events").show(truncate=False)


## BƯỚC 3: CHIẾN LƯỢC 1 - SOFT DELETE (XÓA MỀM BẢO TOÀN TÍNH TOÀN VẸN)

Khi nhận được `op = "d"`, thay vì xóa vật lý dòng dữ liệu, ta dùng `MERGE INTO` để:
- Gán `is_deleted = TRUE`.
- Gán `deleted_at = cdc_time`.
- Cập nhật `_ingest_at = current_timestamp()`.

👉 **Ưu điểm lớn nhất:** Nếu khách hàng `CUST_101` từng có 100 đơn hàng cũ trong bảng `fact_orders`, việc giữ lại dòng này giúp báo cáo doanh thu lịch sử không bị lỗi mất khóa ngoại (Orphan Records).

In [ ]:
# Parse dữ liệu CDC từ Bronze
cdc_schema = StructType([
    StructField("op", StringType(), False),
    StructField("customer_id", StringType(), False),
    StructField("customer_name", StringType(), True),
    StructField("address", StringType(), True),
    StructField("credit_limit", DoubleType(), True),
    StructField("cdc_time", StringType(), True)
])

df_cdc_parsed = spark.table(f"{LH_BRONZE}.cdc_delete_events") \
    .withColumn("data", F.from_json(F.col("raw_payload"), cdc_schema)) \
    .select("data.*") \
    .withColumn("cdc_timestamp", F.to_timestamp("cdc_time"))

# Thực hiện MERGE INTO SOFT DELETE vào bảng Silver
tbl_silver = DeltaTable.forName(spark, f"{LH_SILVER}.customers_delete_demo")

tbl_silver.alias("tgt").merge(
    df_cdc_parsed.alias("src"),
    "tgt.customer_id = src.customer_id"
).whenMatchedUpdate(
    # Nếu là sự kiện XÓA (op = d): Cập nhật trạng thái xóa mềm
    condition = "src.op = 'd'",
    set = {
        "is_deleted": F.lit(True),
        "deleted_at": "src.cdc_timestamp",
        "_ingest_at": F.current_timestamp()
    }
).whenMatchedUpdate(
    # Nếu là sự kiện SỬA (op = u): Cập nhật thông tin bình thường
    condition = "src.op = 'u'",
    set = {
        "customer_name": "src.customer_name",
        "address": "src.address",
        "credit_limit": "src.credit_limit",
        "is_deleted": F.lit(False),
        "_ingest_at": F.current_timestamp()
    }
).execute()

print(f"=== BẢNG {LH_SILVER}.customers_delete_demo SAU KHI XỬ LÝ SOFT DELETE ===")
print("-> Để ý khách hàng CUST_101 đã được chuyển thành is_deleted = true:")
spark.table(f"{LH_SILVER}.customers_delete_demo").show(truncate=False)


## BƯỚC 4: CHIẾN LƯỢC 2 - HARD DELETE (XÓA VẬT LÝ BẰNG DELTA MERGE)

Nếu chính sách doanh nghiệp hoặc luật pháp (như GDPR - Right to be forgotten) bắt buộc phải **xóa vĩnh viễn dòng dữ liệu của khách hàng** ra khỏi Lakehouse:
- Ta sử dụng mệnh đề **`.whenMatchedDelete(condition = "src.op = 'd'")`** trong câu lệnh `MERGE INTO`.

In [ ]:
# Tạo một bảng thử nghiệm riêng cho Hard Delete
spark.table(f"{LH_SILVER}.customers_delete_demo") \
    .write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{LH_SILVER}.customers_hard_delete_demo")

tbl_hard = DeltaTable.forName(spark, f"{LH_SILVER}.customers_hard_delete_demo")

# Lọc riêng sự kiện XÓA của CUST_101 để demo Hard Delete
df_delete_event = df_cdc_parsed.filter("op = 'd'")

tbl_hard.alias("tgt").merge(
    df_delete_event.alias("src"),
    "tgt.customer_id = src.customer_id"
).whenMatchedDelete(
    condition = "src.op = 'd'" # Xóa hẳn bản ghi ra khỏi Delta Table!
).execute()

print(f"=== BẢNG {LH_SILVER}.customers_hard_delete_demo SAU KHI HARD DELETE ===")
print("-> Khách hàng CUST_101 đã biến mất hoàn toàn khỏi bảng:")
spark.table(f"{LH_SILVER}.customers_hard_delete_demo").show(truncate=False)


## BƯỚC 5: TẦNG GOLD (`wh_gold`) TIÊU THỤ DỮ LIỆU ĐÃ XÓA NHƯ THẾ NÀO?

Khi sử dụng **Soft Delete (Khuyên dùng)**, bảng `dim_customer` trên **`wh_gold`** hoặc mô hình **dbt-fabric** chỉ cần lọc:

```sql
-- MODEL DBT HOẶC VIEW TRÊN GOLD WAREHOUSE wh_gold:
SELECT 
    customer_id,
    customer_name,
    address,
    credit_limit
FROM [lh_silver].[dbo].[customers_delete_demo]
WHERE is_deleted = FALSE; -- Tự động loại bỏ khách hàng đã xóa khỏi báo cáo hiện hành!
```

👉 **Kết quả toàn vẹn:**
- Báo cáo khách hàng hiện hành: `CUST_101` không xuất hiện.
- Báo cáo doanh số lịch sử: Các đơn hàng cũ của `CUST_101` vẫn JOIN được tên và địa chỉ cũ của khách mà không bị lỗi mồ côi!